# Chapter 6 — Prompt Engineering: Advanced Techniques · Homework (self-test)

**Goal:** check that you can pick the right prompting lever and implement the control flow around it: few-shot, explicit criteria, evals, retry-with-feedback, chaining, self-correction, output constraints and versioned templates.

- **Theory:** [README.md](README.md) · **Practice:** [01_practice.ipynb](01_practice.ipynb)
- **Estimated time:** 2–2.5 hours
- **Estimated API cost:** $0. Everything scored runs offline against stub models. The one optional live exercise costs about $0.01.
- **Prerequisites:** you have worked through the practice notebook.

### How to use this notebook

1. Attempt everything **without peeking**. Solutions and explanations are at the very bottom, inside collapsed blocks.
2. **Part A — Quiz:** 12 exam-style scenario questions. Put your letters in the `answers` cell and run the grader.
3. **Part B — Coding exercises:** 7 exercises. Replace each `TODO`, then run the check cell under it. Checks never crash the notebook; they print ✓ or a hint.
4. **Part C — Architecture scenario:** write a short design, then compare it with the rubric.
5. **Scorecard:** the overall score is the average of your quiz % and your exercise %. The pass mark is **72%**, which mirrors the Claude exam's 720/1000.

In [ ]:
import hashlib
import json
import os
from typing import Callable

import anthropic
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, ValidationError, model_validator

load_dotenv(find_dotenv())
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
client = anthropic.Anthropic()  # only the optional live exercise uses it


# ---- shared models (same as the practice notebook) ----
class LineItem(BaseModel):
    description: str
    amount: float


class Invoice(BaseModel):
    invoice_number: str | None
    invoice_date: str | None
    purchase_order: str | None
    line_items: list[LineItem]
    total: float


class ValidatedInvoice(Invoice):
    @model_validator(mode="after")
    def total_matches_items(self):
        items_sum = round(sum(i.amount for i in self.line_items), 2)
        if abs(items_sum - self.total) > 0.01:
            raise ValueError(f"total is {self.total} but the line items sum to {items_sum}")
        return self


class StubModel:
    """A deterministic fake "model": replays scripted replies in order and records every prompt it receives."""

    def __init__(self, replies: list):
        self.replies, self.prompts = list(replies), []

    def __call__(self, prompt: str):
        self.prompts.append(prompt)
        assert self.replies, f"the model was called {len(self.prompts)} times, more than scripted (is your loop capped?)"
        return self.replies.pop(0)


LOCAL_RULES = "Review ONE file. Report only issues visible inside this file."
INTEGRATION_RULES = "You receive per-file findings and public interfaces. Report ONLY cross-file issues."

# ---- grading helpers ----
RESULTS: dict[int, bool] = {}
QUIZ_RESULT: tuple[int, int] | None = None


def _run_check(n: int, checks: Callable[[], None]) -> None:
    """Run one exercise's asserts and record the result. Never raises."""
    try:
        checks()
    except NotImplementedError:
        RESULTS[n] = False
        print(f"… Exercise {n}: not implemented yet. Replace the TODO and rerun this cell.")
    except AssertionError as err:
        RESULTS[n] = False
        print(f"✗ Exercise {n} not passed yet: {err}")
    except Exception as err:  # a bug in the solution must not crash the notebook
        RESULTS[n] = False
        print(f"✗ Exercise {n} raised {type(err).__name__}: {err}")
    else:
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")


print("Setup done. Model for the optional live exercise:", MODEL)

In [ ]:
# Only the optional live exercise needs a key. Skip this cell if you have no credit.
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see 00-prerequisites)"

## Part A — Quiz

Each question has exactly one best answer. Distractors are the anti-patterns from [Common mistakes](README.md#common-mistakes-and-anti-patterns).

**Q1.** Your extraction pipeline reads vendor invoices. When the total sits in a table it is extracted correctly. When it appears only in prose ("...for a total of EUR 1,240, payable within 30 days"), the model often returns `null`. The prompt already says "The total may appear anywhere in the document." What is the most effective next step?

- **A)** Rewrite the instruction as "CRITICAL: you MUST ALWAYS find the total."
- **B)** Add 3–5 examples, wrapped in `<example>` tags, that show extraction from both table layouts and prose layouts.
- **C)** Make `total` a required non-null field so the model is forced to return a value.
- **D)** Switch to a model with a larger context window.

**Q2.** A healthcare startup asks Claude Code to "add an audit log for patient-record access". The team has never worked under HIPAA, and nobody has decided the retention period, what counts as "access", or where the logs are stored. What should the architect recommend before any code is written?

- **A)** Write a longer, more detailed task description up front that lists every requirement the team can think of.
- **B)** Let Claude implement a generic audit log now and refactor once requirements are clear.
- **C)** Use the interview pattern: have Claude ask the questions whose answers change the design (retention, scope of "access", storage, tamper resistance) before it plans.
- **D)** Give Claude few-shot examples of audit-log code from an unrelated project.

**Q3.** A security-review bot is prompted with "Be conservative and only report high-confidence issues." It still flags harmless string formatting in log messages as injection, and it missed a real SQL injection in a raw query. What fixes the root cause?

- **A)** Replace the instruction with explicit criteria: flag only when untrusted input reaches a SQL, shell or HTML sink without parameterization or escaping; add a do-not-flag list (log formatting, constants, parameterized queries) and explain why false positives are costly.
- **B)** Ask the model to rate its confidence from 1 to 10 and report only findings above 7.
- **C)** Change the wording to "CRITICAL: you MUST NOT report false positives."
- **D)** Run the review three times and keep only findings that appear at least twice.

**Q4.** Your Pydantic validator rejects an extraction: "total is 427.5 but the line items sum to 412.5". The full document was sent. Which retry request is most likely to produce a correct extraction?

- **A)** Resend the identical request, because output varies between calls.
- **B)** "Your previous answer was wrong. Please try again more carefully."
- **C)** Send only the validation error, without the document, to save tokens.
- **D)** Send the original document, the previous extraction and the specific validation error, and ask for a corrected extraction.

**Q5.** In 4% of insurance claims the `policy_number` is `null`. Investigation shows those claim forms do not contain the policy number at all; it is stored only in the CRM. What should the pipeline do?

- **A)** Retry the extraction up to 3 times with the error "policy_number must not be null".
- **B)** Keep the field nullable, flag the record (or look the value up in the CRM in code), and do not retry the extraction.
- **C)** Add a few-shot example that shows a typical policy-number format.
- **D)** Make `policy_number` a required string in the JSON schema so structured outputs guarantee a value.

**Q6.** Which failure in an extraction pipeline is the right candidate for "retry up to N times with exponential backoff"?

- **A)** Pydantic reports that `invoice_date` is "03/04/25" instead of ISO 8601.
- **B)** The response comes back with `stop_reason: "refusal"`.
- **C)** The API returns HTTP 529 (`overloaded_error`) or the request times out.
- **D)** The purchase-order number is absent from the document.

**Q7.** Which task is the best fit for dynamic decomposition rather than a fixed prompt chain?

- **A)** Reviewing each of 20 changed files, then running one integration pass for cross-file issues.
- **B)** Translating 500 product descriptions with the same glossary.
- **C)** Extracting fields from invoices, validating them and routing exceptions.
- **D)** Investigating why checkout latency doubled after a deploy, where each finding decides what to look at next.

**Q8.** Some supplier invoices contain arithmetic errors: the printed total does not match the line items. Your extractor sometimes silently "corrects" the total, and finance needs to see these cases. Which design is best?

- **A)** Extract `stated_total` and the line items (plus the model's `calculated_total` and `conflict_detected`), recompute the sum in code, and route mismatches to human review.
- **B)** Instruct the model to always return the mathematically correct total.
- **C)** Ask the model for a confidence score and send invoices below 0.8 to review.
- **D)** Retry the extraction until the stated and calculated totals agree.

**Q9.** A draft → review → refine chain produces contract summaries. The review step almost never finds problems, but auditors later find omissions. How should the review step change?

- **A)** Append "Please double-check your work" to the drafting prompt.
- **B)** In the same conversation, ask the drafting model whether its summary is complete.
- **C)** Run the review as a separate call with fresh context that receives only the source contract, the draft and explicit review criteria.
- **D)** Ask the drafting model to write out its full step-by-step reasoning in the response so reviewers can follow it.

**Q10.** You are moving an extraction service to Claude Sonnet 5.5. The old code forced JSON by prefilling the assistant turn with `{`, and elsewhere with `tool_choice: {"type": "tool", "name": "extract"}`. Both now return 400 errors. What is the recommended replacement for schema-conformant JSON?

- **A)** Ask for JSON in the system prompt and parse the reply with a regular expression.
- **B)** Use structured outputs, `client.messages.parse(..., output_format=Model)` (or `output_config.format`), check `stop_reason`, then run semantic validation in code.
- **C)** Set `temperature=0` so the output format is deterministic.
- **D)** Move the `{` prefill from the assistant turn into the system prompt.

**Q11.** A summarization endpoint on Claude Sonnet 5.5 must return summaries of at most about 80 words inside a `<summary>` tag, which a parser reads. To keep summaries short, an engineer set `max_tokens=120`. Now about 15% of responses stop mid-sentence with no closing tag, and the parser crashes on them. What is the best fix?

- **A)** Lower `max_tokens` to 100 so the model learns to be brief.
- **B)** Prefill the assistant turn with `<summary>` so the tag is always present.
- **C)** Ask for the length in the prompt ("at most 80 words, inside `<summary>` tags"), give `max_tokens` a roomy ceiling that also leaves space for thinking, check `stop_reason`, and treat a missing tag as a routed failure.
- **D)** Set `temperature=0` so the length is stable.

**Q12.** Three prompts (invoice, receipt and purchase-order extraction) include the same shared normalization-rules block. To fix a date bug in invoice extraction, you edit that shared block. What does a sound release process require?

- **A)** Bump only the invoice prompt's version, because that is where the bug was.
- **B)** Ship the change and watch production error rates for a week.
- **C)** Copy the block into the invoice prompt so the other two are untouched, and skip the eval because the change is small.
- **D)** Treat all three prompts as changed: each gets a new version ID and hash, runs its own eval set through the regression gate, and gets a changelog entry tied to that eval run before release.

In [ ]:
# Fill in one letter per question: "A", "B", "C" or "D".
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
    "q9": None,
    "q10": None,
    "q11": None,
    "q12": None,
}

In [ ]:
ANSWER_KEY = {
    "q1": "96e2e2c94d02b534d4bdc303373df7b704ed4725981fac17549ee985d132c365",
    "q2": "c85dcf1566dc5c0487a6fe1611c41ba32751e8080b552a419521a39c7fee21b2",
    "q3": "2501e2fe43d858f1773d414ee14ab0a8cdf2afefbc4ffd20ae5d956e618f3ea3",
    "q4": "b634a8333266441f30307b2d3392d77b7b3bcf69bc053cabed2c65d2671fc12e",
    "q5": "fbb43cbbf51767ad8ca178fa6cc4731661f1f1771789b9b7df93fba7ad386e2c",
    "q6": "18cb69e7d9c041e4c1b21fa8c0e110716664b55bef1348d49301f8891c82cef1",
    "q7": "45ff49923feb785985244958470606d87fb75c9912783e470a079e3bb46e1457",
    "q8": "f98cc196593da1da1a7b4550be81b590cd44e7098497c2034a83c78f8a98f3bc",
    "q9": "e19608e84d37f3c94b5f88738c75a8c96772a5e61784e3e24b5512fa65ff63c7",
    "q10": "c69a9867de9de4cc2e45f4cd6a6128833af714d8993edbeac622795c9c5e20d3",
    "q11": "3e23045bd79bd0db2243da95d781972f2a63c00ef2c74914ca8c209b53997f9a",
    "q12": "33d12c43f8f028fb844551369083e2b9daa804b371ce6b8835f63a07a5cac42e",
}  # sha256 of "qid:LETTER", so the answers are not readable here


def grade_quiz(answers: dict) -> None:
    global QUIZ_RESULT
    correct = 0
    for qid, expected_hash in ANSWER_KEY.items():
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) and given.strip() else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == expected_hash
        correct += ok
        print(f"{qid:>3}: {'✓' if ok else '✗'}" + ("  (unanswered)" if letter is None else ""))
    QUIZ_RESULT = (correct, len(ANSWER_KEY))
    print(f"\nQuiz score: {correct}/{len(ANSWER_KEY)} = {correct / len(ANSWER_KEY):.0%}")


grade_quiz(answers)

## Part B — Coding exercises

All seven run offline. `StubModel` (from the setup cell) is a deterministic fake model: it replays scripted replies and records every prompt, which is how you test prompt-control logic without spending tokens.

### Exercise 1 — Few-shot examples as message turns

Implement `build_few_shot_messages(examples, query)`.

- `examples` is a list of `{"input": str, "output": str}` dicts.
- Return a `messages` list for the Messages API: for each example, a `user` turn with the input and an `assistant` turn with the output, in order. Finish with the real `query` as a final `user` turn (a list that ends on an assistant turn is a prefill, which returns a 400 on current models).
- Do not mutate `examples`.
- Raise `ValueError` if `examples` is empty (that would be zero-shot, not few-shot).

[Theory: 6.1](README.md#61-few-shot-prompting)

In [ ]:
def build_few_shot_messages(examples: list[dict], query: str) -> list[dict]:
    # TODO: alternating user/assistant turns from the examples, then the query as a user turn
    raise NotImplementedError

In [ ]:
def check_ex1():
    def checks():
        ex = [{"input": "a", "output": "A"}, {"input": "b", "output": "B"}, {"input": "c", "output": "C"}]
        snapshot = json.dumps(ex)
        msgs = build_few_shot_messages(ex, "d")
        assert isinstance(msgs, list), "return a list of message dicts"
        assert len(msgs) == 7, f"3 examples -> 6 example turns + 1 query turn = 7 messages, got {len(msgs)}"
        roles = [m.get("role") for m in msgs]
        assert roles == ["user", "assistant"] * 3 + ["user"], f"roles must alternate user/assistant and end with user, got {roles}"
        assert [m.get("content") for m in msgs] == ["a", "A", "b", "B", "c", "C", "d"], "contents are missing or in the wrong order"
        assert json.dumps(ex) == snapshot, "do not mutate the examples list"
        try:
            build_few_shot_messages([], "d")
        except ValueError:
            pass
        else:
            raise AssertionError("an empty examples list should raise ValueError (that would be zero-shot)")
    _run_check(1, checks)


check_ex1()

### Exercise 2 — Explicit criteria and a severity rubric

Two parts.

**(a)** Fill in `MY_RUBRIC`: a definition **and** a concrete example for each of `CRITICAL`, `HIGH`, `MEDIUM`, `LOW` (a code-review severity scale). Write criteria, not adjectives: the checker rejects vague words (`appropriate`, `as needed`, `reasonable`, `conservative`, `careful`, `significant`) and shouting (`MUST`, `ALWAYS`, `NEVER`, `IMPORTANT` in capitals). Each definition needs at least 3 words, and the four examples must differ.

**(b)** Implement `build_review_system(categories, do_not_flag, rubric)`. It returns a system prompt containing exactly these lines (blank lines between blocks are fine):

```text
Report findings only in these categories:
- <name>: flag only if <flag_if>          one line per ENABLED category, in dict order
Do not flag: <item1>; <item2>; ...
Severity rubric:
<LEVEL>: <definition> e.g. <example>      one line per rubric level, in dict order
```

Disabled categories must not appear at all.

[Theory: 6.2](README.md#62-explicit-criteria-vs-vague-instructions)

In [ ]:
MY_RUBRIC = {
    "CRITICAL": {"definition": "", "example": ""},  # TODO
    "HIGH": {"definition": "", "example": ""},      # TODO
    "MEDIUM": {"definition": "", "example": ""},    # TODO
    "LOW": {"definition": "", "example": ""},       # TODO
}


def build_review_system(categories: dict, do_not_flag: list[str], rubric: dict) -> str:
    # TODO: render the enabled categories, the do-not-flag list and the rubric in the required line format
    raise NotImplementedError

In [ ]:
import re as _re


def check_ex2():
    def checks():
        cats = {
            "security": {"enabled": True, "flag_if": "untrusted input reaches a query sink"},
            "style": {"enabled": False, "flag_if": "names break the style guide"},
            "correctness": {"enabled": True, "flag_if": "behavior contradicts the docstring"},
        }
        rubric = {"HIGH": {"definition": "Security hole.", "example": "XSS"}, "LOW": {"definition": "Quality.", "example": "duplication"}}
        out = build_review_system(cats, ["style nits", "missing comments"], rubric)
        assert isinstance(out, str), "return the prompt as a string"
        lines = [l.strip() for l in out.splitlines() if l.strip()]
        expected = [
            "Report findings only in these categories:",
            "- security: flag only if untrusted input reaches a query sink",
            "- correctness: flag only if behavior contradicts the docstring",
            "Do not flag: style nits; missing comments",
            "Severity rubric:",
            "HIGH: Security hole. e.g. XSS",
            "LOW: Quality. e.g. duplication",
        ]
        for line in expected:
            assert line in lines, f"missing line: {line!r}"
        assert not any("style:" in l for l in lines), "disabled categories (style) must not appear"
        assert lines.index(expected[1]) < lines.index(expected[2]), "keep the categories in dict order"

        assert set(MY_RUBRIC) == {"CRITICAL", "HIGH", "MEDIUM", "LOW"}, "MY_RUBRIC needs exactly CRITICAL, HIGH, MEDIUM, LOW"
        vague = ("appropriate", "as needed", "reasonable", "conservative", "careful", "significant")
        for level, r in MY_RUBRIC.items():
            d, e = r.get("definition", "").strip(), r.get("example", "").strip()
            assert len(d.split()) >= 3, f"{level}: write a definition of at least 3 words"
            assert e, f"{level}: add a concrete example"
            for word in vague:
                assert word not in (d + " " + e).lower(), f"{level}: {word!r} is an adjective, not a criterion"
            assert not _re.search(r"\b(MUST|ALWAYS|NEVER|IMPORTANT)\b", d + " " + e), f"{level}: no shouting; state the rule calmly"
        examples = [r["example"].strip().lower() for r in MY_RUBRIC.values()]
        assert len(set(examples)) == 4, "use a different example for each level"
        mine = build_review_system(cats, ["style nits"], MY_RUBRIC)
        for level in MY_RUBRIC:
            assert any(l.strip().startswith(f"{level}: ") for l in mine.splitlines()), f"rubric line for {level} missing"
    _run_check(2, checks)


check_ex2()

### Exercise 3 — A tiny eval loop: accuracy, precision, recall

Implement `evaluate(predict, cases, positive="flag")`.

- `cases` is a list of `(text, expected_label)` tuples; `predict(text) -> label` is any function (a Claude call, or a stub).
- Call `predict` **exactly once per case**.
- Return `{"accuracy": ..., "precision": ..., "recall": ...}` as floats in [0, 1], where precision and recall are computed for the `positive` label.
- When a denominator is 0 (for example, nothing was predicted positive), return `0.0` for that metric instead of crashing.

This is the loop you would use to compare a vague prompt with an explicit one, or zero-shot with few-shot.

[Theory: 6.2](README.md#62-explicit-criteria-vs-vague-instructions)

In [ ]:
def evaluate(predict: Callable[[str], str], cases: list[tuple[str, str]], positive: str = "flag") -> dict:
    # TODO: run predict once per case, then compute accuracy, precision and recall for `positive`
    raise NotImplementedError

In [ ]:
def check_ex3():
    def checks():
        cases = [("t0", "flag"), ("t1", "flag"), ("t2", "flag"), ("t3", "ok"), ("t4", "ok"), ("t5", "ok")]

        def close(a, b):
            return abs(a - b) < 1e-9

        def run(pred_map):
            calls = []
            def predict(text):
                calls.append(text)
                return pred_map[text]
            res = evaluate(predict, cases)
            assert len(calls) == len(cases), f"call predict exactly once per case (got {len(calls)} calls for {len(cases)} cases)"
            assert isinstance(res, dict) and {"accuracy", "precision", "recall"} <= set(res), "return a dict with accuracy, precision, recall"
            return res

        perfect = run({f"t{i}": l for i, (_, l) in enumerate(cases)})
        assert all(close(perfect[k], 1.0) for k in ("accuracy", "precision", "recall")), f"perfect predictor should score 1.0 everywhere, got {perfect}"
        allflag = run({f"t{i}": "flag" for i in range(6)})
        assert close(allflag["accuracy"], 0.5) and close(allflag["precision"], 0.5) and close(allflag["recall"], 1.0), f"always-flag: expected acc 0.5, P 0.5, R 1.0, got {allflag}"
        allok = run({f"t{i}": "ok" for i in range(6)})
        assert close(allok["precision"], 0.0) and close(allok["recall"], 0.0), f"never-flag: P and R should be 0.0 (no ZeroDivisionError), got {allok}"
        mixed = run({"t0": "flag", "t1": "ok", "t2": "ok", "t3": "flag", "t4": "ok", "t5": "ok"})
        assert close(mixed["accuracy"], 0.5) and close(mixed["precision"], 0.5) and close(mixed["recall"], 1 / 3), f"mixed: expected acc 0.5, P 0.5, R 0.333, got {mixed}"
    _run_check(3, checks)


check_ex3()

### Exercise 4 — Validate and retry with feedback (stub model)

Implement `extract_with_retry(document, model, max_attempts=3)`. `model(prompt) -> (stop_reason, data)` where `data` is a dict or `None`. Test it with `StubModel` (defined in the setup cell), a deterministic fake that replays scripted replies and records `.prompts`.

1. First prompt: `f"<document>\n{document}\n</document>"`.
2. If `stop_reason != "end_turn"` or `data is None`, return `{"status": "failed", "invoice": None, "attempts": n}` **at once**. Feedback cannot fix a refusal or a cut-off.
3. Validate with `ValidatedInvoice.model_validate(data)`. On success return `{"status": "ok", "invoice": <the ValidatedInvoice>, "attempts": n}`.
4. On `ValidationError`, build the next prompt so that it contains **the original document**, **the previous extraction** (for example `json.dumps(data)`), and **every error message** (`e["msg"] for e in err.errors()`), then try again.
5. After `max_attempts` failed validations, return `{"status": "needs_review", "invoice": None, "attempts": max_attempts}`. Never call the model more than `max_attempts` times.

[Theory: 6.5](README.md#65-validation-and-retry-with-feedback)

In [ ]:
def extract_with_retry(document: str, model: Callable, max_attempts: int = 3) -> dict:
    # TODO: extract -> validate -> retry with document + previous extraction + errors, capped at max_attempts
    raise NotImplementedError


# Try it by hand:
# stub = StubModel([("end_turn", {"invoice_number": "1", "invoice_date": None, "purchase_order": None,
#                                 "line_items": [{"description": "x", "amount": 5}], "total": 5})])
# print(extract_with_retry("doc", stub), stub.prompts)

In [ ]:
def check_ex4():
    def checks():
        doc = "INVOICE INV-42 | Widget A 75.00 | Widget B 70.00 | Shipping 5.00 | TOTAL 150.00"
        items = [{"description": "Widget A", "amount": 75.0}, {"description": "Widget B", "amount": 70.0}]
        good = {"invoice_number": "INV-42", "invoice_date": None, "purchase_order": None,
                "line_items": items + [{"description": "Shipping", "amount": 5.0}], "total": 150.0}
        bad = {"invoice_number": "INV-PREV-42", "invoice_date": None, "purchase_order": None, "line_items": items, "total": 150.0}

        s = StubModel([("end_turn", good)])
        r = extract_with_retry(doc, s)
        assert isinstance(r, dict) and r.get("status") == "ok", f"valid first reply should give status 'ok', got {r}"
        assert r.get("attempts") == 1 and len(s.prompts) == 1, "valid first reply: exactly 1 attempt and 1 call"
        assert isinstance(r.get("invoice"), ValidatedInvoice), "return the validated ValidatedInvoice object under 'invoice'"
        assert doc in s.prompts[0], "the first prompt must contain the document"

        s = StubModel([("end_turn", bad), ("end_turn", good)])
        r = extract_with_retry(doc, s)
        assert r.get("status") == "ok" and r.get("attempts") == 2, f"bad then good should be ok after 2 attempts, got {r}"
        retry = s.prompts[1]
        assert doc in retry, "the retry prompt must include the original document"
        assert "INV-PREV-42" in retry, "the retry prompt must include the previous extraction"
        assert "145" in retry, "the retry prompt must include the specific validation error (items sum to 145)"

        s = StubModel([("end_turn", bad)] * 5)
        r = extract_with_retry(doc, s)
        assert r.get("status") == "needs_review" and r.get("invoice") is None, f"always-invalid should end in needs_review, got {r}"
        assert len(s.prompts) == 3 and r.get("attempts") == 3, f"default cap is 3 calls, got {len(s.prompts)}"
        s = StubModel([("end_turn", bad)] * 5)
        extract_with_retry(doc, s, max_attempts=2)
        assert len(s.prompts) == 2, "respect max_attempts"

        for stop in ("max_tokens", "refusal"):
            s = StubModel([(stop, None), ("end_turn", good)])
            r = extract_with_retry(doc, s)
            assert r.get("status") == "failed" and len(s.prompts) == 1, f"stop_reason {stop!r} must fail at once, without a retry"
    _run_check(4, checks)


check_ex4()

### Exercise 5 — Prompt chain: per-file passes plus an integration pass

Implement `run_review_chain(files, interfaces, model)`. `model(system, user) -> str` (a stub here; a Claude call in production).

1. For each `(path, source)` in `files`, in order, call `model(LOCAL_RULES, user)` where `user` is `f'<file path="{path}">\n{source}\n</file>'`. One file per call: no other file's source.
2. Then make **one** integration call, `model(INTEGRATION_RULES, user)`, where `user` contains `f'<findings file="{path}">\n{findings}\n</findings>'` for every file plus `f"<interfaces>\n{interfaces}\n</interfaces>"`. The integration pass sees findings and interfaces, **not** the full sources.
3. Return `{"local": {path: findings, ...}, "integration": <integration text>}`.

`LOCAL_RULES` and `INTEGRATION_RULES` are defined in the setup cell.

[Theory: 6.3](README.md#63-prompt-chaining)

In [ ]:
def run_review_chain(files: dict[str, str], interfaces: str, model: Callable[[str, str], str]) -> dict:
    # TODO: one local call per file, then one integration call over the findings + interfaces
    raise NotImplementedError

In [ ]:
def check_ex5():
    def checks():
        files = {"db.py": "SRC_DB = 1", "auth.py": "SRC_AUTH = 2", "api.py": "SRC_API = 3"}
        interfaces = "IFACE: get_user(id) -> dict | None"
        calls = []

        def model(system, user):
            calls.append((system, user))
            return f"RESULT-{len(calls)}"

        out = run_review_chain(files, interfaces, model)
        assert len(calls) == 4, f"3 files -> 3 local calls + 1 integration call = 4, got {len(calls)}"
        for (system, user), (path, src) in zip(calls[:3], files.items()):
            assert system == LOCAL_RULES, "local calls must use LOCAL_RULES as the system prompt"
            assert src in user and f'path="{path}"' in user, f"local call for {path} must contain that file and its path"
            others = [s for p, s in files.items() if p != path]
            assert not any(o in user for o in others), f"local call for {path} must not include other files (attention dilution)"
        system, user = calls[3]
        assert system == INTEGRATION_RULES, "the last call must be the integration pass with INTEGRATION_RULES"
        for i, path in enumerate(files, start=1):
            assert f"RESULT-{i}" in user and f'file="{path}"' in user, f"integration input must include the findings for {path}"
        assert interfaces in user, "integration input must include the interfaces"
        assert not any(src in user for src in files.values()), "integration pass should see findings + interfaces, not full sources"
        assert isinstance(out, dict), "return a dict"
        assert out.get("local") == {"db.py": "RESULT-1", "auth.py": "RESULT-2", "api.py": "RESULT-3"}, "out['local'] must map path -> findings"
        assert out.get("integration") == "RESULT-4", "out['integration'] must be the integration call's text"
    _run_check(5, checks)


check_ex5()

### Exercise 6 — Self-check routing: recompute in code

Implement `route_invoice(inv)`. `inv` is a dict extracted with self-correction fields:
`line_items` (list of `{"amount": float, ...}`), `stated_total` (float or `None`), `conflict_detected` (the **model's** claim, bool) and `purchase_order` (str or `None`).

Rules, in this order:
1. Recompute the line-item sum **in code**. If `stated_total` is not `None` and differs from the sum by more than 0.01, return `"human_review"`. Ignore `conflict_detected`: the model can be wrong in either direction.
2. Else, if `purchase_order` is `None`, return `"request_document"` (the PO lives in another document; do not retry).
3. Else return `"accept"`.

[Theory: 6.6](README.md#66-self-correction)

In [ ]:
def route_invoice(inv: dict) -> str:
    # TODO: recompute the sum in code; never trust conflict_detected alone
    raise NotImplementedError

In [ ]:
def check_ex6():
    def checks():
        def inv(amounts, stated, conflict, po):
            return {"line_items": [{"description": f"i{n}", "amount": a} for n, a in enumerate(amounts)],
                    "stated_total": stated, "conflict_detected": conflict, "purchase_order": po}

        cases = [
            (inv([75, 70], 150, True, "PO-1"), "human_review", "stated 150 vs items 145"),
            (inv([75, 70], 150, False, "PO-1"), "human_review", "model claims no conflict, but the numbers disagree"),
            (inv([75, 70], 145, True, "PO-1"), "accept", "model claims a conflict, but the numbers agree"),
            (inv([10], None, False, "PO-1"), "accept", "no stated total printed"),
            (inv([1200], 1200, False, None), "request_document", "consistent, but the PO is missing"),
            (inv([75, 70], 150, True, None), "human_review", "a conflict takes priority over a missing PO"),
            (inv([0.1, 0.2], 0.3, False, "PO-1"), "accept", "float rounding: 0.1 + 0.2 should match 0.3"),
        ]
        for data, expected, why in cases:
            got = route_invoice(data)
            assert got == expected, f"{why}: expected {expected!r}, got {got!r}"
    _run_check(6, checks)


check_ex6()

### Exercise 7 — Strict template rendering and a traceable prompt version

Implement two helpers for prompts managed as versioned templates.

1. `render(template, variables)` replaces every `{{NAME}}` placeholder (letters, digits, underscore) with `variables["NAME"]`.
   - Raise `ValueError` if a placeholder has no variable (missing) **or** a variable has no placeholder (unused).
   - Insert values **literally**: backslashes such as `C:\new\1` must survive, and a value that itself contains `{{X}}` is not rendered again.
   - Leave single braces alone, so JSON examples like `{"label": "billing"}` in the template still work.
2. `prompt_sha(model, system, user_template)` returns the first 12 hex characters of the SHA-256 of `f"{model}\n{system}\n{user_template}"`. The model is part of the hash, because the same text on another model is a different version.

[Theory: 6.11](README.md#611-prompts-as-versioned-modular-templates-expanded-beyond-the-guide)

In [ ]:
def render(template: str, variables: dict[str, str]) -> str:
    # TODO: strict {{NAME}} rendering (missing or unused -> ValueError, values inserted literally)
    raise NotImplementedError


def prompt_sha(model: str, system: str, user_template: str) -> str:
    # TODO: first 12 hex chars of sha256(f"{model}\n{system}\n{user_template}")
    raise NotImplementedError

In [ ]:
def check_ex7():
    def checks():
        assert render("Hi {{NAME}}, {{NAME}}!", {"NAME": "Ann"}) == "Hi Ann, Ann!", "replace every occurrence"
        for bad, why in [({}, "a missing variable"), ({"NAME": "Ann", "TIER": "gold"}, "an unused variable")]:
            try:
                render("Hi {{NAME}}", bad)
            except ValueError:
                pass
            else:
                raise AssertionError(f"{why} must raise ValueError")
        assert render("<p>{{P}}</p>", {"P": r"C:\new\1"}) == r"<p>C:\new\1</p>", "backslashes must stay literal"
        assert render("{{A}}", {"A": "{{B}}"}) == "{{B}}", "do not render a value a second time"
        json_tpl = 'Example: {"label": "billing"}\n<ticket>{{T}}</ticket>'
        assert render(json_tpl, {"T": "x"}) == 'Example: {"label": "billing"}\n<ticket>x</ticket>', "single braces must survive"
        sha = prompt_sha("claude-sonnet-5-5", "sys", "user")
        expected = hashlib.sha256("claude-sonnet-5-5\nsys\nuser".encode()).hexdigest()[:12]
        assert sha == expected, "hash the model, system and user template joined by newlines; keep 12 hex characters"
        assert prompt_sha("claude-opus-5-5", "sys", "user") != sha, "a different model must give a different version"
    _run_check(7, checks)


check_ex7()

### Optional live exercise (live — calls the API, ~$0.01, not scored)

Use your `build_few_shot_messages` from Exercise 1 to classify a ticket with Claude. Compare the label with and without the examples.

In [ ]:
EXAMPLES = [
    {"input": "My blender came with a cracked jar. Refund please.", "output": "<label>damaged</label>"},
    {"input": "WHERE IS MY ORDER?? 2 weeks and nothing!!!", "output": "<label>shipping</label>"},
    {"input": "Please change the card I pay with.", "output": "<label>billing</label>"},
]
SYSTEM = "Classify the ticket into one queue: billing, damaged, shipping, account, human. Reply only with <label>...</label>."
ticket = "The lamp works but the shade is cracked and I'd like my money back."

runs = [("zero-shot", [{"role": "user", "content": ticket}])]
try:
    runs.append(("few-shot", build_few_shot_messages(EXAMPLES, ticket)))
except NotImplementedError:
    print("Finish Exercise 1 first; running zero-shot only.")

for name, messages in runs:
    response = client.messages.create(model=MODEL, max_tokens=1024, system=SYSTEM, messages=messages)
    text = "".join(b.text for b in response.content if b.type == "text")
    print(f"{name:<10} stop_reason={response.stop_reason}  {text.strip()}")

## Part C — Architecture scenario

**Scenario.** An insurer wants a claims-intake service. About 30,000 documents a day arrive in many layouts: structured PDF forms, scanned letters and forwarded emails. Each claim needs: claimant name, policy number, incident date, a list of claimed amounts plus the stated total, and a short description. The policy number is often missing from the claim itself; it lives in the CRM. Downstream, the payments system **auto-pays claims under $500** when every check passes; everything else goes to an adjuster.

Design the extraction part of this system. Cover:

1. Prompt design: what goes in the system prompt (criteria, normalization rules, examples), and why.
2. Schema and validation layers: what the structured output guarantees, and what your code must check.
3. Retry policy: which failures you retry (and with what), which you never retry, and the cap.
4. Self-correction fields and routing outcomes.
5. How you evaluate a prompt change before you roll it out.
6. Where the $500 auto-pay rule is enforced.

Write your answer in the cell below (about 200–300 words).

*Your answer:*

<details><summary>Rubric: what a strong answer contains (open after you have written yours)</summary>

- **Few-shot for layouts:** 3–5 relevant, diverse examples in `<example>` tags covering form, letter and email layouts, including one with the total only in prose and one with no policy number.
- **Explicit normalization rules:** ISO 8601 dates (relative dates resolved against a supplied today-date), amounts as number + ISO 4217 currency, `null` for anything absent, "never guess". Calm, precise wording, no shouting.
- **Structured outputs** (`messages.parse` / `output_config.format`) for syntax, with `policy_number` **nullable**. A `stop_reason` check before trusting `parsed_output`; `max_tokens` and `refusal` are failures, not retries.
- **Semantic validation in code (Pydantic validators):** date format, amounts positive, `calculated_total` equals the item sum, `conflict_detected` consistent.
- **Retry policy:** retry once or twice with the document + previous extraction + exact error for fixable errors (format, misplaced field, arithmetic slip). **Never** retry for the missing policy number: look it up in the CRM in code, or flag it. Exponential backoff only for 429/529/timeouts. A hard cap, then human review.
- **Self-correction:** `stated_total` vs `calculated_total` + `conflict_detected`, recomputed in Python; conflicts route to an adjuster, never auto-pay. Optionally a `detected_pattern` / reason field to group failures.
- **Routing outcomes:** accept, request document / CRM lookup, human review, failed.
- **Evaluation:** a labeled set of real documents with expected fields and routes; compare prompt variants on accuracy (and false auto-pay rate) before rollout; keep watching dismissal and override rates by pattern after rollout.
- **The $500 rule is enforced in code** (a deterministic check before payment), not in the prompt. Prompts are probabilistic; business rules that must always hold belong in code.
- Bonus: the per-document chain is fixed and code-owned (extract → validate → self-check → route), and the Message Batches API could cut cost for this non-urgent volume (Chapter 7).

</details>

## Scorecard

In [ ]:
quiz_correct, quiz_total = QUIZ_RESULT if QUIZ_RESULT else (0, 12)
ex_total = 7
ex_passed = sum(RESULTS.get(n, False) for n in range(1, ex_total + 1))
quiz_pct, ex_pct = quiz_correct / quiz_total, ex_passed / ex_total
overall = (quiz_pct + ex_pct) / 2

print(f"Quiz       : {quiz_correct}/{quiz_total} ({quiz_pct:.0%})" + ("" if QUIZ_RESULT else "  <- run the quiz grader"))
print(f"Exercises  : {ex_passed}/{ex_total} passed ({ex_pct:.0%})   " +
      " ".join(f"ex{n}:{'✓' if RESULTS.get(n) else '✗'}" for n in range(1, ex_total + 1)))
print(f"Overall    : {overall:.0%}  ->  {'PASS' if overall >= 0.72 else 'RETRY'} (pass mark 72%)")

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers and explanations</summary>

**Q1: B.** Extraction from different layouts is a textbook few-shot case: examples show the mapping that an instruction already failed to convey. A is shouting, which current models over-trigger on, and it adds no information. C forces a value even when there is none, which invites invention. D does not address the cause; the documents already fit in context.

**Q2: C.** The failure mode is unknown requirements in an unfamiliar, regulated domain, which is exactly what the interview pattern surfaces. A assumes the team already knows every consideration, and it does not. B builds the wrong thing and pays for a rewrite. D teaches a pattern from a different domain (examples must be relevant).

**Q3: A.** "Conservative" is an adjective, not a criterion. Explicit flag / do-not-flag criteria fix both the false positive and the miss. B is unreliable because the model can be confidently wrong. C is shouting and still undefined. D suppresses real bugs that are detected inconsistently.

**Q4: D.** A useful retry has all three ingredients: the source to re-read, the answer to correct, and the exact discrepancy. A and B give the model nothing to correct against. C removes the source the model needs to re-check the amounts.

**Q5: B.** A retry cannot find information that is not in the document; it only burns tokens or pushes the model to invent a value. A is exactly that. C teaches a format for a value that is absent. D forces invention at the schema level.

**Q6: C.** Backoff is for transient infrastructure errors, where the same request will probably succeed later. A is a semantic error: retry once with specific feedback, not with backoff. B will not change by waiting. D cannot be fixed by any retry.

**Q7: D.** Dynamic decomposition fits when the steps emerge from intermediate results. An open-ended investigation is the classic case. A, B and C have steps known in advance, so a code-owned chain is more predictable, testable and cheaper.

**Q8: A.** Self-correction fields make the conflict visible to code, and recomputing in Python means you do not depend on the model's flag. B hides exactly the cases finance needs. C relies on self-rated confidence, which is unreliable. D can never succeed for a document that contradicts itself, and it pressures the model to change a stated value.

**Q9: C.** A reviewer inside its own generation context is less likely to challenge earlier decisions. An independent call with explicit criteria catches more. A and B keep the same context. D is the reasoning-extraction anti-pattern: on current models it can be declined with `stop_reason: "refusal"`.

**Q10: B.** Structured outputs constrain the reply to your schema on current models; the schema guarantees syntax, and your validators add semantics. A gives no guarantee. C fails twice: current models reject a non-default temperature with a 400, and temperature never guaranteed a format. D is not a prefill at all, so it guarantees nothing.

**Q11: C.** `max_tokens` is a budget, not a length instruction: it truncates mid-sentence (`stop_reason: "max_tokens"`), and thinking tokens count toward it too. Length belongs in the prompt, and defensive parsing turns a bad reply into a routed failure instead of a crash. A makes truncation worse. B returns a 400 on Claude 4.6+ models. D returns a 400 on current models (and a `TypeError` in the Python SDK v1.0+), and never controlled length.

**Q12: D.** A shared block changes what every including prompt sends, so every one of them is a new version that must pass its own eval; the hash makes that visible. A leaves two changed prompts untested and untraceable. C could be a deliberate design choice, but skipping the eval is not. B tests in production, after customers are affected.

</details>

<details><summary>Exercise 1 — reference solution</summary>

```python
def build_few_shot_messages(examples: list[dict], query: str) -> list[dict]:
    if not examples:
        raise ValueError("few-shot needs at least one example")
    messages = []
    for ex in examples:
        messages.append({"role": "user", "content": ex["input"]})
        messages.append({"role": "assistant", "content": ex["output"]})
    messages.append({"role": "user", "content": query})  # end on a user turn: no prefill
    return messages
```

</details>

<details><summary>Exercise 2 — reference solution</summary>

```python
MY_RUBRIC = {
    "CRITICAL": {"definition": "Users hit a runtime failure in production.", "example": "null dereference while processing a payment"},
    "HIGH": {"definition": "Security vulnerability an attacker can exploit.", "example": "SQL injection, XSS, missing authorization check"},
    "MEDIUM": {"definition": "Logic bug with no immediate user impact.", "example": "wrong sort order, off-by-one in pagination"},
    "LOW": {"definition": "Code quality issue that does not change behavior.", "example": "duplicated helper, slow algorithm on tiny inputs"},
}


def build_review_system(categories: dict, do_not_flag: list[str], rubric: dict) -> str:
    lines = ["Report findings only in these categories:"]
    lines += [f"- {name}: flag only if {c['flag_if']}" for name, c in categories.items() if c["enabled"]]
    lines.append("Do not flag: " + "; ".join(do_not_flag))
    lines.append("Severity rubric:")
    lines += [f"{level}: {r['definition']} e.g. {r['example']}" for level, r in rubric.items()]
    return "\n".join(lines)
```

</details>

<details><summary>Exercise 3 — reference solution</summary>

```python
def evaluate(predict: Callable[[str], str], cases: list[tuple[str, str]], positive: str = "flag") -> dict:
    preds = [predict(text) for text, _ in cases]
    labels = [label for _, label in cases]
    tp = sum(p == positive and l == positive for p, l in zip(preds, labels))
    fp = sum(p == positive and l != positive for p, l in zip(preds, labels))
    fn = sum(p != positive and l == positive for p, l in zip(preds, labels))
    return {
        "accuracy": sum(p == l for p, l in zip(preds, labels)) / len(cases),
        "precision": tp / (tp + fp) if tp + fp else 0.0,
        "recall": tp / (tp + fn) if tp + fn else 0.0,
    }
```

</details>

<details><summary>Exercise 4 — reference solution</summary>

```python
def extract_with_retry(document: str, model: Callable, max_attempts: int = 3) -> dict:
    prompt = f"<document>\n{document}\n</document>"
    for attempt in range(1, max_attempts + 1):
        stop_reason, data = model(prompt)
        if stop_reason != "end_turn" or data is None:
            return {"status": "failed", "invoice": None, "attempts": attempt}
        try:
            return {"status": "ok", "invoice": ValidatedInvoice.model_validate(data), "attempts": attempt}
        except ValidationError as err:
            errors = "\n".join(f"- {e['msg']}" for e in err.errors())
            prompt = (f"<document>\n{document}\n</document>\n"
                      f"<previous_extraction>\n{json.dumps(data, indent=2)}\n</previous_extraction>\n"
                      f"<validation_errors>\n{errors}\n</validation_errors>\n"
                      "Your previous extraction failed validation. Re-read the document and correct it.")
    return {"status": "needs_review", "invoice": None, "attempts": max_attempts}
```

</details>

<details><summary>Exercise 5 — reference solution</summary>

```python
def run_review_chain(files: dict[str, str], interfaces: str, model: Callable[[str, str], str]) -> dict:
    local = {path: model(LOCAL_RULES, f'<file path="{path}">\n{source}\n</file>') for path, source in files.items()}
    bundle = "\n".join(f'<findings file="{p}">\n{f}\n</findings>' for p, f in local.items())
    integration = model(INTEGRATION_RULES, f"{bundle}\n<interfaces>\n{interfaces}\n</interfaces>")
    return {"local": local, "integration": integration}
```

</details>

<details><summary>Exercise 6 — reference solution</summary>

```python
def route_invoice(inv: dict) -> str:
    recomputed = round(sum(item["amount"] for item in inv["line_items"]), 2)
    stated = inv.get("stated_total")
    if stated is not None and abs(recomputed - stated) > 0.01:
        return "human_review"
    if inv.get("purchase_order") is None:
        return "request_document"
    return "accept"
```

</details>

<details><summary>Exercise 7 — reference solution</summary>

```python
import re

VAR = re.compile(r"{{(\w+)}}")


def render(template: str, variables: dict[str, str]) -> str:
    names = set(VAR.findall(template))
    missing, unused = names - variables.keys(), variables.keys() - names
    if missing or unused:
        raise ValueError(f"missing={sorted(missing)} unused={sorted(unused)}")
    return VAR.sub(lambda m: variables[m.group(1)], template)  # single pass; a function keeps backslashes literal


def prompt_sha(model: str, system: str, user_template: str) -> str:
    return hashlib.sha256(f"{model}\n{system}\n{user_template}".encode()).hexdigest()[:12]
```

</details>

<details><summary>Part C — sample answer outline</summary>

Fixed per-document chain owned by code: **extract** with `messages.parse` (nullable `policy_number`; fields `stated_total`, `calculated_total`, `conflict_detected`; system prompt with normalization rules and 4 layout examples) → **check `stop_reason`** → **validate** in Pydantic (ISO date, positive amounts, arithmetic) → **retry once or twice** with document + previous extraction + exact error → **CRM lookup in code** for a missing policy number → **recompute the total in Python**: conflict → adjuster → **route** (auto-pay candidate, adjuster, request document, failed). The `< $500` rule is a deterministic check in the payments service. Prompt changes ship only after they beat the current prompt on a labeled set of real claims, measured by field accuracy and false auto-pay rate.

</details>

## Done

- Below 72%? Reread the README sections behind your ✗ marks, rerun the matching practice cells, and retake.
- Passed? Check the box in the [README](README.md) learning objectives.

Next chapter: [Chapter 7 — Message Batches API](../07-message-batches/README.md).